# TP5 — Firewall, IDS y Honeypot

**ARyS · IF046 · UNPSJB Trelew — Unidad 5**

Laboratorio reproducible con Docker: red bridge `10.0.0.0/24` con IPs estáticas.

| Contenedor | IP | Rol |
|---|---|---|
| `servidor` | `10.0.0.10` | nftables + servicio web + Suricata IDS |
| `honeypot` | `10.0.0.20` | Cowrie (SSH/Telnet honeypot) |
| `atacante` | `10.0.0.30` | nmap, curl, sshpass |

Todas las celdas de código ejecutan comandos `docker exec` contra esos contenedores
y capturan su salida real. Las celdas Markdown responden cada consigna teórica de
las Partes A, B, C y D, más un apéndice con los temas opcionales.

> Arranque previo (ver `README.md`): `docker compose up -d --build`.
> Ejecutar todo y grabar salidas: `jupyter nbconvert --to notebook --execute --inplace TP5_Laboratorio.ipynb`.

In [1]:
import os, shutil, subprocess

def run(script):
    """Ejecuta comandos shell (docker exec, docker compose, etc.) e imprime su salida.

    Se usa subprocess en lugar de `!` / `%%bash` para que la salida quede siempre
    capturada en la celda y para que el CLI de Docker no consuma el stdin del kernel
    (esto último pasa con docker.exe bajo WSL 2).
    """
    p = subprocess.run(
        script, shell=True, executable="/bin/bash",
        stdin=subprocess.DEVNULL, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True,
    )
    print(p.stdout, end="" if p.stdout.endswith("\n") else "\n")
    return p.returncode

print("Directorio:", os.getcwd())
print("CLI docker:", shutil.which("docker") or "NO ENCONTRADO (activar integracion WSL / instalar Docker)")

Directorio: /home/gonza/tp5rys
CLI docker: /tmp/opencode/bin/docker


In [2]:
run("""
docker --version
docker compose ps
""")

Docker version 29.8.2, build 7fc2dff
NAME       IMAGE                  COMMAND                  SERVICE    CREATED          STATUS          PORTS
atacante   tp5-lab-atacante       "sleep infinity"         atacante   22 minutes ago   Up 22 minutes   
honeypot   cowrie/cowrie:latest   "/cowrie/cowrie-env/…"   honeypot   18 minutes ago   Up 18 minutes   2222-2223/tcp
servidor   tp5-lab-servidor       "sleep infinity"         servidor   22 minutes ago   Up 22 minutes   


0

---

## Parte A — Firewall con nftables

Objetivo: construir una cadena `input` con **default deny** e inspección de estado,
y comprobar qué pasa con un puerto no habilitado.

In [3]:
run("""
# Estado inicial y limpieza para que el notebook sea idempotente.
docker exec servidor nft flush ruleset 2>/dev/null || true
docker exec servidor bash -lc "pkill -f '^python3 -m http.server' 2>/dev/null || true"
docker exec servidor bash -lc "pkill -f '^suricata ' 2>/dev/null || true"
docker exec servidor /usr/sbin/sshd 2>/dev/null || true
echo "=== ruleset inicial ==="
docker exec servidor nft list ruleset
""")

=== ruleset inicial ===


0

In [4]:
run("""
docker exec servidor nft add table inet filtro
docker exec servidor nft add chain inet filtro entrada '{ type filter hook input priority 0; policy drop; }'
docker exec servidor nft add rule inet filtro entrada ct state established,related accept
docker exec servidor nft add rule inet filtro entrada iif lo accept
docker exec servidor nft add rule inet filtro entrada tcp dport 22 accept
echo "=== ruleset aplicado ==="
docker exec servidor nft list ruleset
""")

=== ruleset aplicado ===
table inet filtro {
	chain entrada {
		type filter hook input priority filter; policy drop;
		ct state established,related accept
		iif "lo" accept
		tcp dport 22 accept
	}
}


0

In [5]:
run("""
echo "=== prueba: nmap -Pn -p 22,8080 10.0.0.10 desde el atacante ==="
docker exec atacante nmap -Pn -p 22,8080 10.0.0.10
echo
echo "=== prueba: SSH al puerto 22 (si esta permitido) ==="
docker exec atacante bash -lc "sshpass -p 'tp5lab' ssh -o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null -o ConnectTimeout=5 root@10.0.0.10 'echo SSH_OK; whoami' 2>&1 || true"
""")

=== prueba: nmap -Pn -p 22,8080 10.0.0.10 desde el atacante ===
Starting Nmap 7.93 ( https://nmap.org ) at 2026-11-18 21:13 UTC
Nmap scan report for servidor.tp5-lab_lab_net (10.0.0.10)
Host is up (0.000047s latency).

PORT     STATE    SERVICE
22/tcp   open     ssh
8080/tcp filtered http-proxy
MAC Address: 0A:08:B2:24:ED:36 (Unknown)

Nmap done: 1 IP address (1 host up) scanned in 1.37 seconds

=== prueba: SSH al puerto 22 (si esta permitido) ===
SSH_OK
root


0

### Respuestas — Parte A

**A1. ¿Qué pasa si intentás conectarte al puerto 8080 con la política en drop?**

El escaneo previo lo muestra: el 8080 aparece como **filtered**, mientras el 22
aparece **open** y el SSH funciona. El paquete SYN al 8080 no coincide con ninguna
regla `accept`: no es `established` (es el primer paquete de una conexión nueva,
estado `NEW`), no entra por `lo`, y no tiene `dport 22`. Al no haber regla que lo
acepte, se aplica la política de cadena `drop` y el paquete se descarta
silenciosamente. Por eso nmap dice *filtered* y no *closed*: no llega ningún RST
de vuelta que indique “puerto cerrado”, simplemente no hay respuesta.

**A2. ¿Qué hace `ct state established,related accept` y por qué es stateful?**

`ct` accede a **conntrack**, la tabla de conexiones del kernel. El primer paquete
de una conexión entrante es `NEW` y lo acepta la regla explícita `tcp dport 22
accept`; a partir de ahí, todos los paquetes de esa misma sesión son
`established`, y los flujos auxiliares asociados son `related`. La regla los
acepta sin volver a evaluar una lista de puertos. Esa **memoria de la sesión** es
exactamente lo que distingue al firewall de **inspección de estado** del
filtrado de paquetes *stateless*, que decide paquete por paquete mirando sólo
cabeceras de capas 3–4 (IP, puerto, protocolo) y que por sí solo no sabría que
la respuesta de un SSH ya autorizado debe pasar. Es la categoría intermedia de
la clasificación por capas; por encima queda la inspección de aplicación (capa 7,
base de NGFW/WAF). Sin esta regla, el handshake del SSH se cortaría después del
primer SYN aceptado.

**A3. Default deny / menor privilegio**

La cadena nace con `policy drop`: se bloquea todo y cada `accept` es una
excepción mínima y explícita (loopback, SSH y —en la Parte B— el servicio
publicado). Eso es una **lista blanca**, alineada con el principio de menor
privilegio: si se olvida una regla, el tráfico se cae (fail-safe), a diferencia
de `default allow`, donde cualquier olvido deja un puerto abierto.

---

## Parte B — Publicar un servicio con DNAT

Objetivo: levantar `python3 -m http.server` en el servidor y publicarlo en el
puerto 80, y analizar el riesgo que introduce esa regla.

In [6]:
run("""
docker exec -d servidor python3 -m http.server 8000 --directory /srv/www
sleep 1
echo "=== servicio interno ==="
docker exec servidor bash -lc "ps aux | grep '[h]ttp.server' || true"
docker exec servidor nft add chain inet filtro prerouting '{ type nat hook prerouting priority -100; policy accept; }'
docker exec servidor nft add rule inet filtro prerouting tcp dport 80 dnat ip to 10.0.0.10:8000
# Ajuste necesario: el paquete ya DNATeado debe poder pasar por la cadena input.
# 'ct original proto-dst 80' lo distingue de un acceso directo al puerto 8000.
docker exec servidor nft add rule inet filtro entrada tcp dport 8000 ct original proto-dst 80 accept
echo "=== cadena prerouting (NAT) ==="
docker exec servidor nft list chain inet filtro prerouting
echo
echo "=== cadena entrada (filter) ==="
docker exec servidor nft list chain inet filtro entrada
""")

=== servicio interno ===
root         787  3.5  0.2  26592 19756 ?        Ss   21:13   0:00 python3 -m http.server 8000 --directory /srv/www
=== cadena prerouting (NAT) ===
table inet filtro {
	chain prerouting {
		type nat hook prerouting priority dstnat; policy accept;
		tcp dport 80 dnat ip to 10.0.0.10:8000
	}
}

=== cadena entrada (filter) ===
table inet filtro {
	chain entrada {
		type filter hook input priority filter; policy drop;
		ct state established,related accept
		iif "lo" accept
		tcp dport 22 accept
		tcp dport 8000 ct original proto-dst 80 accept
	}
}


0

In [7]:
run("""
echo "=== curl http://10.0.0.10/ (puerto publico 80) ==="
docker exec atacante curl -i --max-time 5 http://10.0.0.10/ || true
echo
echo "=== acceso directo al puerto 8000 (debe quedar filtered) ==="
docker exec atacante nmap -Pn -p 8000 10.0.0.10
""")

=== curl http://10.0.0.10/ (puerto publico 80) ===
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0HTTP/1.0 200 OK
Server: SimpleHTTP/0.6 Python/3.12.3
Date: Wed, 18 Nov 2026 21:13:12 GMT
Content-type: text/html
Content-Length: 127
Last-Modified: Wed, 18 Nov 2026 20:49:28 GMT

<!doctype html>
<title>TP5</title>
<h1>TP5 - servicio publicado con DNAT</h1>
<p>Servidor: 10.0.0.10 - puerto interno 8000</p>

100   127  100   127    0     0  41530      0 --:--:-- --:--:-- --:--:-- 42333

=== acceso directo al puerto 8000 (debe quedar filtered) ===
Starting Nmap 7.93 ( https://nmap.org ) at 2026-11-18 21:13 UTC
Nmap scan report for servidor.tp5-lab_lab_net (10.0.0.10)
Host is up (0.000039s latency).

PORT     STATE    SERVICE
8000/tcp filtered http-alt
MAC Address: 0A:08:B2:24:ED:36 (Unknown)

Nmap done: 1 I

0

### Respuestas — Parte B

**B1. Verificación del acceso por el puerto 80**

`curl http://10.0.0.10/` devuelve `HTTP/1.0 200 OK` y el HTML del servicio
interno: el paquete llega al hook `prerouting`, la regla DNAT reescribe el
destino `10.0.0.10:80 → 10.0.0.10:8000`, y luego la regla de `input` con
`ct original proto-dst 80` lo deja pasar. En cambio, `nmap -p 8000` sigue marcando
el puerto como *filtered*: no se abrió el 8000 al exterior, sólo se publicó el
80. (Nota de implementación: el enunciado original no incluía esa regla de
`input`; sin ella, el paquete DNATeado era droppeado por la política `drop`, así
que se agregó esa aceptación acotada al puerto original 80.)

**B2. Análisis de riesgo de cada DNAT**

Una regla DNAT es una puerta de entrada desde la red menos confiable: publica un
servicio que antes no era alcanzable. Si ese servicio es comprometido, el
atacante queda ejecutándose **dentro de la zona donde vive el servicio**. Si el
destino está en la **DMZ**, el daño queda confinado a la subred perimetral y hay
otro firewall entre la DMZ y la LAN: hay que volver a atravesar un punto de
control para llegar a los equipos internos (defensa en profundidad). Si el DNAT
apuntara directamente a la LAN, un solo exploit convertiría al servidor
publicado en trampolín para movimiento lateral hacia toda la red interna. Por eso
las reglas DNAT deben minimizarse y preferentemente apuntar a DMZ, idealmente
con un proxy reverso/WAF delante.

---

## Parte C — Detección con un IDS (Suricata)

Objetivo: ejecutar Suricata en modo IDS sobre `eth0`, repetir el `nmap -sS -sV`
de la Unidad 3 y leer las alertas.

In [8]:
run("""
docker exec servidor mkdir -p /var/log/suricata
docker exec servidor bash -lc "pkill -f '^suricata ' 2>/dev/null || true"
# Limpiar logs para que las alertas mostradas correspondan solo a esta ejecucion.
docker exec servidor bash -lc ": > /var/log/suricata/fast.log; : > /var/log/suricata/eve.json; : > /var/log/suricata/suricata.log"
docker exec -d servidor suricata -i eth0 -c /etc/suricata/suricata.yaml -l /var/log/suricata -D
# Esperar a que termine de cargar 53k reglas y arrive a "Engine started".
for i in $(seq 1 90); do
  if docker exec servidor grep -q 'Engine started' /var/log/suricata/suricata.log 2>/dev/null; then
    echo "Suricata listo tras ${i}s"
    break
  fi
  sleep 1
done
echo "=== proceso Suricata ==="
docker exec servidor bash -lc "ps aux | grep '[s]uricata' || true"
echo
echo "=== log de arranque ==="
docker exec servidor bash -lc "tail -n 12 /var/log/suricata/suricata.log 2>/dev/null || true"
""")

Suricata listo tras 17s
=== proceso Suricata ===
root         863 95.6  7.7 2370156 629648 ?      Ssl  21:13   0:16 suricata -i eth0 -c /etc/suricata/suricata.yaml -l /var/log/suricata -D

=== log de arranque ===
[857 - Suricata-Main] 2026-11-18 21:13:13 Info: suricata: Setting engine mode to IDS mode by default
[857 - Suricata-Main] 2026-11-18 21:13:13 Info: exception-policy: master exception-policy set to: auto
[857 - Suricata-Main] 2026-11-18 21:13:13 Info: ioctl: eth0: MTU 1500
[863 - Suricata-Main] 2026-11-18 21:13:13 Info: logopenfile: fast output device (regular) initialized: fast.log
[863 - Suricata-Main] 2026-11-18 21:13:13 Info: logopenfile: eve-log output device (regular) initialized: eve.json
[863 - Suricata-Main] 2026-11-18 21:13:13 Info: logopenfile: stats output device (regular) initialized: stats.log
[863 - Suricata-Main] 2026-11-18 21:13:20 Info: detect: 2 rule files processed. 53503 rules successfully loaded, 0 rules failed, 0
[863 - Suricata-Main] 2026-11-18 21:13:20

0

In [9]:
run("""
echo "=== ataque desde el atacante: nmap -sS -sV 10.0.0.10 ==="
docker exec atacante nmap -sS -sV 10.0.0.10
""")

=== ataque desde el atacante: nmap -sS -sV 10.0.0.10 ===
Starting Nmap 7.93 ( https://nmap.org ) at 2026-11-18 21:13 UTC
Nmap scan report for servidor.tp5-lab_lab_net (10.0.0.10)
Host is up (0.000077s latency).
Not shown: 998 filtered tcp ports (no-response)
PORT   STATE SERVICE VERSION
22/tcp open  ssh     OpenSSH 9.6p1 Ubuntu 3ubuntu13.20 (Ubuntu Linux; protocol 2.0)
80/tcp open  http    SimpleHTTPServer 0.6 (Python 3.12.3)
MAC Address: 0A:08:B2:24:ED:36 (Unknown)
Service Info: OS: Linux; CPE: cpe:/o:linux:linux_kernel

Service detection performed. Please report any incorrect results at https://nmap.org/submit/ .
Nmap done: 1 IP address (1 host up) scanned in 10.94 seconds


0

In [10]:
run("""
echo "=== /var/log/suricata/fast.log ==="
docker exec servidor tail -n 80 /var/log/suricata/fast.log 2>/dev/null || echo "(fast.log todavia no existe)"
echo
echo "=== eventos alert en eve.json ==="
docker exec servidor grep -F '"event_type":"alert"' /var/log/suricata/eve.json 2>/dev/null | tail -n 12 || true
""")

=== /var/log/suricata/fast.log ===
11/18/2026-21:13:36.190945  [**] [1:9000001:1] TP5 LOCAL SCAN - SYN scan (many SYNs from one source) [**] [Classification: (null)] [Priority: 3] {TCP} 10.0.0.30:46822 -> 10.0.0.10:5959
11/18/2026-21:13:36.193112  [**] [1:9000001:1] TP5 LOCAL SCAN - SYN scan (many SYNs from one source) [**] [Classification: (null)] [Priority: 3] {TCP} 10.0.0.30:46822 -> 10.0.0.10:1296
11/18/2026-21:13:36.193162  [**] [1:9000001:1] TP5 LOCAL SCAN - SYN scan (many SYNs from one source) [**] [Classification: (null)] [Priority: 3] {TCP} 10.0.0.30:46822 -> 10.0.0.10:5004
11/18/2026-21:13:36.195313  [**] [1:9000001:1] TP5 LOCAL SCAN - SYN scan (many SYNs from one source) [**] [Classification: (null)] [Priority: 3] {TCP} 10.0.0.30:46822 -> 10.0.0.10:1271
11/18/2026-21:13:36.195368  [**] [1:9000001:1] TP5 LOCAL SCAN - SYN scan (many SYNs from one source) [**] [Classification: (null)] [Priority: 3] {TCP} 10.0.0.30:46822 -> 10.0.0.10:1272
11/18/2026-21:13:36.197542  [**] [1:9000

0

### Respuestas — Parte C

**C1. ¿Detectó el nmap?**

Sí. En `fast.log` aparecen alertas de escaneo (reglas *ET SCAN* de Emerging
Threats Open descargadas con `suricata-update`, más la regla local
`TP5 LOCAL SCAN - SYN scan` de respaldo): la mayor cantidad de SYNs desde la IP
`10.0.0.30` en pocos segundos dispara el umbral. `eve.json` guarda los mismos
eventos en JSON para el SIEM. Cabe aclarar que Suricata captura en `eth0` con
AF_PACKET, es decir **antes** del hook `input` de nftables: por eso ve los SYNs
aunque el firewall los descarte después.

**C2. IDS vs. IPS**

Lo hecho es un **IDS**: Suricata observa una copia del tráfico, alerta y no
toca el flujo. El escaneo terminó igual; el firewall por su parte filtró los
puertos, pero Suricata no los bloqueó. En modo **IPS inline** (por ejemplo con
reglas `drop` y NFQUEUE o af-packet en modo IPS, con Suricata colocado en el
camino del tráfico) el paquete sospechoso se podría descartar en el momento, a
costo de convertir al sensor en un punto de falla y en un punto en el camino de
los datos. En esta topología el sensor está en el propio servidor (endpoint),
así que no está en el camino del resto de las máquinas: para proteger a todo el
segmento haría falta un gateway con Suricata inline.

**C3. ¿Firmas o anomalías?**

Por **firmas**. Las reglas comparan patrones conocidos: umbrales de SYN desde un
mismo origen, strings del banner/probe de Nmap, etc. No se construyó un modelo
de comportamiento “normal” ni se midió un desvío estadístico, que sería
detección por anomalías. Las firmas dan pocos falsos positivos pero son ciegas
ante un 0-day o un escaneo con firma nueva; por eso los sistemas modernos
combinan ambos enfoques y se apoyan en metadatos/fingerprinting (JA3/JA4) cuando
el tráfico va cifrado.

---

## Parte D — Honeypot (Cowrie)

Objetivo: atacar el honeypot SSH y leer la inteligencia que produce Cowrie.

In [11]:
run("""
echo "=== intentos SSH contra el honeypot 10.0.0.20 ==="
docker exec atacante bash -lc 'for p in password 123456 qwerty tp5lab; do echo "--- password probada: $p"; sshpass -p "$p" ssh -o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null -o ConnectTimeout=5 root@10.0.0.20 "id; uname -a; ls -la /tmp" 2>&1 || true; done'
""")

=== intentos SSH contra el honeypot 10.0.0.20 ===
--- password probada: password
uid=0(root) gid=0(root) groups=0(root)
Linux svr04 6.1.0-21-amd64 #1 SMP PREEMPT_DYNAMIC Debian 6.1.90-1 (2024-05-03) x86_64 GNU/Linux
drwxrwxrwt 1 root root 4096 2026-05-04 14:46 .
drwxr-xr-x 1 root root    0 2024-06-12 00:00 ..
--- password probada: 123456
Permission denied, please try again.
--- password probada: qwerty
uid=0(root) gid=0(root) groups=0(root)
Linux svr04 6.1.0-21-amd64 #1 SMP PREEMPT_DYNAMIC Debian 6.1.90-1 (2024-05-03) x86_64 GNU/Linux
drwxrwxrwt 1 root root 4096 2026-05-04 14:46 .
drwxr-xr-x 1 root root    0 2024-06-12 00:00 ..
--- password probada: tp5lab
uid=0(root) gid=0(root) groups=0(root)
Linux svr04 6.1.0-21-amd64 #1 SMP PREEMPT_DYNAMIC Debian 6.1.90-1 (2024-05-03) x86_64 GNU/Linux
drwxrwxrwt 1 root root 4096 2026-05-04 14:46 .
drwxr-xr-x 1 root root    0 2024-06-12 00:00 ..


0

In [12]:
run(r"""
# La imagen oficial de Cowrie 3.x es distroless: no trae shell ni tail.
# Se lee el log con el python que viene en el propio contenedor.
echo "=== eventos relevantes de cowrie.json ==="
docker exec honeypot /cowrie/cowrie-env/bin/python3 -c '
import json
p = "/cowrie/cowrie-git/var/log/cowrie/cowrie.json"
eventos = []
for linea in open(p, encoding="utf-8"):
    try:
        e = json.loads(linea)
    except ValueError:
        continue
    if e.get("eventid") in ("cowrie.session.connect", "cowrie.client.version",
                            "cowrie.login.failed", "cowrie.login.success",
                            "cowrie.command.input"):
        eventos.append(e)
for e in eventos[-25:]:
    print(e.get("timestamp"), e.get("eventid"),
          "src=" + str(e.get("src_ip")),
          "user=" + str(e.get("username")),
          "pass=" + str(e.get("password")),
          "input=" + str(e.get("input")))
'
echo
echo "=== ultimas lineas crudas ==="
docker exec honeypot /cowrie/cowrie-env/bin/python3 -c 'print("".join(open("/cowrie/cowrie-git/var/log/cowrie/cowrie.json").readlines()[-8:]))'
""")

=== eventos relevantes de cowrie.json ===
2026-11-18T21:12:29.706377Z cowrie.client.version src=10.0.0.30 user=None pass=None input=None
2026-11-18T21:12:29.712005Z cowrie.login.failed src=10.0.0.30 user=root pass=123456 input=None
2026-11-18T21:12:30.719666Z cowrie.session.connect src=10.0.0.30 user=None pass=None input=None
2026-11-18T21:12:30.719836Z cowrie.client.version src=10.0.0.30 user=None pass=None input=None
2026-11-18T21:12:30.725129Z cowrie.login.success src=10.0.0.30 user=root pass=qwerty input=None
2026-11-18T21:12:30.726825Z cowrie.command.input src=10.0.0.30 user=None pass=None input=id; uname -a; ls -la /tmp
2026-11-18T21:12:30.738217Z cowrie.session.connect src=10.0.0.30 user=None pass=None input=None
2026-11-18T21:12:30.738387Z cowrie.client.version src=10.0.0.30 user=None pass=None input=None
2026-11-18T21:12:30.743468Z cowrie.login.success src=10.0.0.30 user=root pass=tp5lab input=None
2026-11-18T21:12:30.745042Z cowrie.command.input src=10.0.0.30 user=None pass=N

0

### Respuestas — Parte D

**D1. Qué registró Cowrie**

El log muestra la IP de origen `10.0.0.30`, el usuario `root`, cada contraseña
probada (`password`, `123456`, `qwerty`, `tp5lab`), los eventos de
`login failed` / `login successful` y, cuando la sesión falsa se acepta, los
comandos ejecutados (`id`, `uname -a`, `ls -la /tmp`). Cowrie 3.x registra eso en
`/cowrie/cowrie-git/var/log/cowrie/cowrie.json` (el output por defecto es
`jsonlog`, no el `cowrie.log` de versiones anteriores): cada línea trae
`eventid`, `timestamp`, `src_ip`, `username`, `password`, `input` e incluso el
fingerprint HASSH del cliente. Es **alta señal**: sabemos quién escaneó, con qué
credenciales y qué intentó hacer, sin riesgo real para un sistema productivo.

**D2. ¿Por qué casi cero falsos positivos?**

Un honeypot es un recurso señuelo que no presta ningún servicio legítimo:
**nadie** debería conectarse a él. Cada interacción es intrínsecamente
sospechosa, a diferencia de un IDS, donde una firma puede dispararse con tráfico
cotidiano. No hay usuarios ni aplicaciones legítimas que deban iniciar sesión en
esa máquina; cualquier conexión es, por definición, reconocimiento o ataque. Por
eso complementa tan bien al IDS: pocos eventos pero de altísima confianza.

**D3. Honeytoken propuesto**

Sembrar un archivo `config.ini` señuelo en un recurso compartido con una
credencial falsa, por ejemplo:

```
[reporting]
user = svc_reporting
password = N0tARealP@ss-2026
host = 10.0.0.99
```

Esa credencial no existe en ningún sistema real: es un honeytoken. La alerta
dispararía cuando alguien la usa: un evento de autenticación con
`user=svc_reporting` en cualquier equipo, o una conexión SSH desde un host del
laboratorio hacia `10.0.0.99`, se correlaciona en el SIEM (o con una regla de
Suricata/Cowrie) y genera una alerta crítica con la IP del usuario del token.
También se podría sembrar una fila falsa en una base de datos o un señuelo en un
repositorio Git; la idea es siempre la misma: artefacto falso + monitoreo de su
uso.

---

## Apéndice — Temas opcionales (“Para investigar”, suma)

**1. Migrar una regla de iptables a nftables**

`iptables -A INPUT -m conntrack --ctstate ESTABLISHED,RELATED -j ACCEPT` se
escribe en nftables como `nft add rule inet filtro entrada ct state
established,related accept`. Cambian la sintaxis (único comando `nft` para
IPv4/IPv6/ARP/bridge, en lugar de `iptables`, `ip6tables`, etc.), la estructura
(tablas → cadenas → reglas con hooks explícitos como `input` o `prerouting`) y
el backend: las distribuciones modernas ya ejecutan `iptables-nft`, es decir
la sintaxis vieja se traduce a reglas nftables por debajo.

Fuente: nftables (proyecto netfilter) — https://wiki.nftables.org

**2. Qué agrega un NGFW y por qué no tiene definición normativa**

Sobre un firewall stateful, un NGFW agrega inspección de capa 7 (DPI),
identidad de usuario, control de aplicaciones e IPS integrado. Es un término de
industria acuñado por **Gartner**, no aparece en ningún NIST SP ni RFC, por lo
que conviene citarlo como concepto de mercado y no como estándar. En cambio el
**WAF** sí tiene una referencia práctica concreta: el OWASP Core Rule Set (CRS
4.x), que inspecciona HTTP/HTTPS para mitigar el OWASP Top 10.

Fuentes: Gartner Glossary (NGFW) — https://www.gartner.com/en/information-technology/glossary/next-generation-firewalls-ngfws · OWASP CRS — https://coreruleset.org/

**3. Cómo cambia el diseño bajo Zero Trust (NIST SP 800-207)**

El modelo “castillo y foso” daba por confiable todo lo que estuviera “adentro”.
Zero Trust elimina esa confianza implícita por ubicación de red: **cada acceso
se autentica, autoriza y evalúa por sesión**, se asume que el atacante ya puede
estar dentro. En la práctica reemplaza al “adentro confiable” por
**microsegmentación** (segmentos mínimos con política estricta entre ellos) y,
en entornos cloud-native, por firewalling basado en **identidad de workload**
mediante eBPF (p. ej. Cilium en Kubernetes) en vez de reglas estáticas por IP o
VLAN. En este laboratorio la bridge plana `10.0.0.0/24` es justo el modelo
clásico; un diseño Zero Trust dividiría servidor, honeypot y atacante en
segmentos separados con políticas explícitas entre cada par.

Fuentes: NIST SP 800-207 (2020) y SP 800-207A (2023) — https://csrc.nist.gov/pubs/sp/800/207/final · Cilium — https://cilium.io/

---

## Cierre

Las salidas capturadas arriba son la evidencia del laboratorio. Para entregar:

```bash
jupyter nbconvert --to pdf TP5_Laboratorio.ipynb --output TP5_ApellidoNombre.pdf
# alternativa sin LaTeX:
jupyter nbconvert --to html TP5_Laboratorio.ipynb --output TP5_ApellidoNombre.html
```

Para levantar el entorno desde cero: `docker compose up -d --build`.
Para desarmarlo: `docker compose down`.